In [ ]:
import sys, pathlib

# Ensure flat, root-anchored imports work from notebooks
try:
    sys.path.append(str(pathlib.Path(__file__).resolve().parents[1]))
    ROOT = pathlib.Path(__file__).resolve().parents[1]
except NameError:
    # Jupyter fallback: ascend from CWD until repo root is found
    ROOT = pathlib.Path.cwd()
    if not ((ROOT / "configs").exists() and (ROOT / "utils").exists()):
        probe = ROOT
        for _ in range(6):
            if ((probe / "configs").exists() and (probe / "utils").exists()):
                ROOT = probe
                break
            probe = probe.parent
        # Final fallback: add CWD anyway
        sys.path.append(str(ROOT))
    else:
        sys.path.append(str(ROOT))

print("Repo root:", ROOT)


In [ ]:
from pathlib import Path
import yaml
import json
import os
import numpy as np
import random as pyrandom

from utils.plotting import set_matplotlib_style
from utils.run import RunContext
import utils.config as ucfg
from stages.diagnostics import run_diagnostics

try:
    from utils.seeds import set_global_seeds
except Exception:
    set_global_seeds = None

set_matplotlib_style()


In [ ]:
# Load and validate configuration (default + diagnostics)
cfg_paths = [ROOT / "configs" / "default.yaml", ROOT / "configs" / "diagnostics.yaml"]
schema_path = ROOT / "configs" / "schema.json"
for p in cfg_paths + [schema_path]:
    if not p.exists():
        raise FileNotFoundError(f"Missing config file: {p}")

cfg = None
_loader_called = False
for fname in [
    "load_and_validate",
    "load_config",
    "load_cfg",
    "load_configs",
    "load_yaml_with_schema",
    "load_yaml",
]:
    if hasattr(ucfg, fname):
        func = getattr(ucfg, fname)
        try:
            try:
                cfg = func(cfg_paths, schema_path=schema_path)
                _loader_called = True
                break
            except TypeError:
                try:
                    cfg = func(cfg_paths, schema=schema_path)
                    _loader_called = True
                    break
                except TypeError:
                    cfg = func(cfg_paths)
                    _loader_called = True
                    break
        except Exception as e:
            print(f"Config loader {fname} failed: {e}")

if not _loader_called or cfg is None:
    # Manual deep-merge fallback then validate if possible
    from collections import deque
    cfg = {}
    for p in cfg_paths:
        with open(p, "r", encoding="utf-8") as fh:
            data = yaml.safe_load(fh) or {}
        dq = deque()
        dq.append((cfg, data))
        while dq:
            tgt, src = dq.popleft()
            if not isinstance(src, dict):
                continue
            for k, v in src.items():
                if isinstance(v, dict):
                    if k not in tgt or not isinstance(tgt.get(k), dict):
                        tgt[k] = {}
                    dq.append((tgt[k], v))
                else:
                    tgt[k] = v
    validated = False
    for vname in [
        "validate",
        "validate_config",
        "validate_cfg",
        "validate_against_schema",
    ]:
        if hasattr(ucfg, vname):
            try:
                vfunc = getattr(ucfg, vname)
                try:
                    vfunc(cfg, schema_path=schema_path)
                except TypeError:
                    vfunc(cfg, schema=schema_path)
                validated = True
                break
            except Exception as e:
                print(f"Validation via {vname} failed: {e}")
    if not validated:
        print("Warning: Falling back without explicit validation; downstream stages may validate.")

print("Loaded config keys (sample):", sorted(list(cfg.keys()))[:12])

# Deterministic seeds
seed = None
for path in [
    ("diagnostics", "seed"),
    ("diagnostics", "random_seed"),
    ("seeds", "global"),
    ("seed",),
    ("random_seed",),
]:
    try:
        if len(path) == 2:
            val = cfg.get(path[0], {}).get(path[1], None)
        else:
            val = cfg.get(path[0], None)
        if isinstance(val, (int, np.integer)):
            seed = int(val)
            break
    except Exception:
        pass
if seed is None:
    seed = 123

if set_global_seeds is not None:
    set_global_seeds(seed)
else:
    np.random.seed(seed)
    pyrandom.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except Exception:
        pass
print("Seed set to:", seed)


In [ ]:
# Run diagnostics stage
run = RunContext.start(cfg)
ctx = run.stage("diagnostics")
print("Running stage: diagnostics")
run_diagnostics(cfg, ctx)

# Finalize run
try:
    run.close(inputs=[str(p) for p in cfg_paths], notes="orchestrated by notebooks/05_diagnostics.ipynb")
except Exception as e:
    print("run.close warning:", e)

# Minimal acceptance: confirm at least one figure and one table artifact exist
fig_dir = ROOT / "results" / "diagnostics" / "figures"
tab_dir = ROOT / "results" / "diagnostics" / "tables"
n_fig = 0
n_tab = 0
if fig_dir.exists():
    n_fig = len(list(fig_dir.glob("*.png"))) + len(list(fig_dir.glob("*.pdf")))
if tab_dir.exists():
    n_tab = len(list(tab_dir.glob("*.csv"))) + len(list(tab_dir.glob("*.tex")))
print(f"Diagnostics artifacts → figures: {n_fig}, tables: {n_tab}")
if n_fig == 0 or n_tab == 0:
    print("Warning: Expected at least one figure and one table for diagnostics stage.")
